> **교육 목표** 시계열을 표로 바꿔 4일차 회귀 모델(랜덤포레스트)로 예측하고, 미래 정보 누수(lag 1)를 피합니다.
>
> **핵심 내용** `lag7`·`lag14`·`roll7`·요일 피처 → 시간 순 분할 → 기준선·SARIMA와 비교 → 변수 중요도 → lag 1의 함정

In [1]:
%%html
<style>
table {
    float: left;
    clear: both;
}
</style><!-- Markdown 왼쪽 정렬 -->

# D5_06 ML로 시계열 예측하기

> **오늘 질문:** 시계열을 '표'로 바꾸면 4일차 회귀 모델로도 예측할 수 있을까?

**규칙:** 다음 7일을 예측하므로, 피처는 **7일 이상 지난 값만** 씁니다. (lag 7 이상)

| 피처 | 뜻 |
|---|---|
| `lag7`, `lag14` | 7일 전, 14일 전 사용량 (같은 요일) |
| `roll7` | 7일 전부터 거슬러 7일간 평균 (최근 수준) |
| `dow`, `weekend` | 요일(0=월), 주말 여부 |

| 표시 | 의미 |
|---|---|
| 📋 제공 | 코드가 들어 있습니다. 실행만 하세요 |
| ✍️ 입력 | 설명을 보고 직접 입력하세요 |
| 💬 해석 | 결과를 보고 답을 적어 보세요 |

## 0. 준비 📋

In [ ]:
%matplotlib inline
import pandas as pd
import matplotlib.pyplot as plt
import koreanize_matplotlib
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error

# 1교시에서 한 시간 정리 (00:00 기록 바로잡기 → 구간 시작 시각으로)
se = pd.read_csv("../../data/steel_energy.csv")
se["date"] = pd.to_datetime(se["date"], format="%d/%m/%Y %H:%M")
midnight = (se["date"].dt.hour == 0) & (se["date"].dt.minute == 0)
se.loc[midnight, "date"] += pd.Timedelta(days=1)
se["date"] -= pd.Timedelta(minutes=15)
se = se.set_index("date").sort_index()
daily = se["usage_kwh"].resample("D").sum()
cut = int(len(daily) * 0.8)

## 1. 시계열을 표로 ✍️

`shift(7)`은 값을 7칸 아래로 밀어 '7일 전 값'을 만듭니다. `rolling`은 **먼저 7칸 민 다음** 계산해야 미래가 섞이지 않습니다.

> 변수 이름: `df`, `features`

In [ ]:
# 힌트: 시리즈.shift(7), 시리즈.shift(7).rolling(7).mean(), 인덱스.dayofweek

## 2. 시간 순 분할과 학습 ✍️

앞 20일은 과거 값이 없어(NaN) 학습에서 뺍니다. Test 날의 피처는 모두 7일 이상 지난 실제 값이라, 매주 새로 학습하지 않아도 Walk-forward와 같은 조건이 됩니다.

> 변수 이름: `train`, `test`, `rf`, `pred`

In [ ]:
# 힌트: 표.iloc[:cut].dropna(), RandomForestRegressor(n_estimators=300, min_samples_leaf=2, random_state=42)

## 3. 기준선과 비교 📋

같은 Test 기간의 계절 naive(= lag7 그대로)와 비교합니다.

In [ ]:
snaive_mae = mean_absolute_error(test["usage"], test["lag7"])
rf_mae = mean_absolute_error(test["usage"], pred)
print(f"계절 naive MAE {snaive_mae:.0f} kWh")
print(f"랜덤포레스트 MAE {rf_mae:.0f} kWh  (계절 naive 대비 {(1 - rf_mae / snaive_mae) * 100:.1f}% 개선)")

## 4. 변수 중요도와 예측 그래프 ✍️

## 5. lag 1의 함정 📋

'어제 사용량(lag1)'을 넣으면 어떻게 될까요? 숫자는 좋아지지만, 다음 주 예측에는 쓸 수 없는 피처입니다.

In [ ]:
df["lag1"] = df["usage"].shift(1)
features_leak = features + ["lag1"]
train_leak = df.iloc[:cut].dropna()
rf_leak = RandomForestRegressor(n_estimators=300, min_samples_leaf=2, random_state=42)
rf_leak.fit(train_leak[features_leak], train_leak["usage"])
pred_leak = rf_leak.predict(df.iloc[cut:][features_leak])
print(f"lag1 포함 MAE {mean_absolute_error(test['usage'], pred_leak):.0f} kWh  ← 다음 주 예측에서는 불가능한 점수")

## 정리 💬

**Q1.** 랜덤포레스트는 계절 naive와 SARIMA(5교시, 약 660 kWh)에 비해 어떤가요? 가장 중요한 피처는?

> 답:


**Q2.** lag1을 넣은 모델의 MAE는 왜 '불가능한 점수'일까요?

> 답:
